# 02 - Spanning Oriented Subspaces

This notebook is the canonical Chapter 2 study build for *Geometric Algebra for Computer Science*. It is standalone: the explanations, computations, visual artifacts, and checks here are original teaching material for the chapter's mathematical territory rather than copied textbook prose.

**Verified source span.** I checked the local PDF before writing. Printed p. 23 is PDF page 55 and opens Chapter 2, "Spanning Oriented Subspaces". Printed p. 64 is PDF page 96 and is the final page of Chapter 2; PDF page 97 opens Chapter 3, "Metric Products of Subspaces". The extracted headings in the span include vector spaces; oriented line, area, and volume elements; scalars; applications; homogeneous subspace representation; Grassmann algebra; outer-product properties; exercises; drawing bivectors; hidden surface removal; and singularity detection.


## Translation Guide

Chapter 2 is deliberately nonmetric: it builds subspaces with the outer product before any dot product, norm, or angle is allowed to do work. In this notebook, coordinates are used only for drawing and checking.

| Geometric idea | Algebraic object | Coordinate stand-in used here |
|---|---|---|
| Oriented line through the origin | vector `a` | a 2-D or 3-D NumPy vector |
| Oriented area element | bivector `a wedge b` | signed determinant in 2-D, or `(e12, e23, e31)` in 3-D |
| Oriented volume element | trivector `a wedge b wedge c` | determinant of the three spanning vectors |
| Direct subspace representation | blade `A` with `x wedge A = 0` for contained vectors | small symbolic multivectors from `utils.ga` |
| General Grassmann element | multivector with grade parts | dictionary-backed `Multivector` values |
| Programming labs | drawing and testing orientation | Plotly/Matplotlib artifacts saved under `artifacts/chapter-02` |


## Notebook Route

1. Set up the algebra and artifact paths.
2. Build oriented lines, bivectors, trivectors, and scalars from the outer product.
3. Use bivector ratios for linear equations and planar line intersections.
4. Treat blades as direct subspace representations and separate blades from general k-vectors.
5. Practice Grassmann algebra operations: grade selection, reversion, and grade involution.
6. Run the three programming labs: drawing bivectors, backface culling, and singularity detection.
7. Finish with invariant checks that should survive small changes to the numeric examples.


In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, IFrame, display

PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "utils" / "ga").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CHAPTER_DIR = Path.cwd()
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "chapter-02"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

from utils import chapter02_subspaces as ch2
from utils.artifacts import save_json, save_matplotlib, save_plotly_html
from utils.ga import Algebra

np.set_printoptions(precision=4, suppress=True)


def show_html(path: Path, *, height: int = 720) -> None:
    display(IFrame(src=Path(path).resolve().as_uri(), width="100%", height=height))


def show_png(path: Path, *, width: int = 920) -> None:
    uri = Path(path).resolve().as_uri()
    display(HTML(f'<img src="{uri}" style="max-width: {width}px; width: 100%; height: auto;" />'))


source_span = {
    "printed_pages": "23-64",
    "pdf_pages": "55-96",
    "boundary_check": "PDF page 55 starts Chapter 2; PDF page 97 starts Chapter 3.",
}
source_span


{'printed_pages': '23-64',
 'pdf_pages': '55-96',
 'boundary_check': 'PDF page 55 starts Chapter 2; PDF page 97 starts Chapter 3.'}

## How To Read The Chapter Without A Metric

The unusual discipline in this chapter is that length, angle, perpendicularity, and normal vectors are not allowed to explain the main constructions. That can feel artificial at first because most computer-graphics and linear-algebra habits reach for dot products immediately. Here the point is to isolate a different question: what can be known just from spanning and orientation? A surprising amount survives. Parallelness is visible because repeated directions make a wedge vanish. Area ratios inside the same plane are visible because any two full-grade area elements in that plane differ only by a scalar. Volume ratios inside one 3-D carrier are visible for the same reason.

The terms attitude, orientation, and weight are the notebook's compass. Attitude is the carrier subspace: which line, plane, or volume direction the element occupies. Orientation is the sign convention on that carrier: reversing a vector factor reverses a line, swapping two factors reverses an area, and odd permutations reverse a volume. Weight is the relative scalar amount inside the same attitude. None of these require a Euclidean ruler. The metric chapter that follows will add comparisons between different attitudes; Chapter 2 is the cleaner pre-metric layer that makes those later comparisons meaningful.


## The Outer Product As A Spanning Operation

A vector already carries more than a bare set of points on a line: it has a carrier line, an orientation, and a scalar weight along that line. Chapter 2 asks for the same kind of object for planes and volumes. The outer product is the operation that promotes "spanned by these vectors" into a first-class algebraic value.

The crucial rules are compact: the outer product is linear, associative, and antisymmetric on vector factors. The antisymmetry makes repeated vector factors vanish, which is exactly what we want when vectors fail to span new dimensions.


In [2]:
algebra3 = Algebra([1, 1, 1], names=["e1", "e2", "e3"])
e1, e2, e3 = algebra3.basis()
zero3 = algebra3.scalar(0)

a = algebra3.vector([1.2, 0.4, 0.0])
b = algebra3.vector([0.2, 1.1, 0.3])
c = algebra3.vector([0.4, -0.2, 1.3])

B = a.wedge(b)
T_left = a.wedge(b).wedge(c)
T_right = a.wedge(b.wedge(c))

print("a wedge b =", B)
print("(a wedge b) wedge c =", T_left)
print("a wedge (b wedge c) =", T_right)

assert a.wedge(a).almost_equal(zero3)
assert a.wedge(b).almost_equal(-b.wedge(a))
assert T_left.almost_equal(T_right)


a wedge b = 1.24e1^e2 + 0.36e1^e3 + 0.12e2^e3
(a wedge b) wedge c = 1.732e1^e2^e3
a wedge (b wedge c) = 1.732e1^e2^e3


## Lines, Bivectors, And Trivectors In One Scene

The next artifact shows the same build-up visually. A line element is drawn as an oriented vector. The bivector is the oriented area swept by `a` and `b`; its drawn parallelogram is only one possible representative of the area element. The trivector is the oriented volume swept by adding `c`.


In [3]:
scene_a = np.array([1.35, 0.35, 0.05])
scene_b = np.array([0.15, 1.05, 0.35])
scene_c = np.array([0.35, -0.25, 1.15])

fig = ch2.subspace_scene(scene_a, scene_b, scene_c)
scene_path = save_plotly_html(
    fig,
    "figures",
    "oriented-lines-bivectors-trivectors",
    "subspace-scene.html",
    root=ARTIFACT_ROOT,
)
print(scene_path)
show_html(scene_path)


D:\Geometry\artifacts\chapter-02\figures\oriented-lines-bivectors-trivectors\subspace-scene.html


## What The Picture Is Supposed To Teach

The Plotly scene is deliberately not a proof. Its job is to train your eye to stop treating a bivector as a decorated vector normal. In 3-D Euclidean drawing, a normal vector is a convenient visual proxy for a plane, but it smuggles in a metric: perpendicularity is not part of the outer product itself. The actual bivector is the oriented plane element, and its coordinates belong to the basis areas `e12`, `e23`, and `e31`. The Euclidean dual normal appears only because the screen needs something easy to display.

The same warning applies to the transparent volume. A trivector in ordinary 3-D has only one possible attitude, so its data collapses to a signed scalar multiple of the pseudoscalar. That does not make it an ordinary scalar. It still carries the unit volume element as part of its meaning. If the basis orientation changes, the coefficient changes with it. Keeping the trivector as a grade-3 object prevents bookkeeping mistakes that become painful in higher-dimensional models, especially the homogeneous and conformal models used later for actual Euclidean geometry.


## Bivectors Are Reshapeable, Not Anchored

A bivector remembers the plane attitude, orientation, and weighted area. It does not remember a preferred corner, a preferred rectangle, or even the particular vector pair that produced it. That is why the same `a wedge b` can be drawn as many sheared parallelograms or as a disk-like patch in the same oriented plane.

This is not a loss of information for the subspace computation. It is exactly the abstraction that lets the algebra solve area-ratio problems without committing to coordinates too early.


In [4]:
grid_fig = ch2.bivector_grid_figure(rows=4, cols=6)
grid_path = save_matplotlib(
    grid_fig,
    "figures",
    "bivector-drawing-grid",
    "bivector-grid.png",
    root=ARTIFACT_ROOT,
    dpi=170,
)
plt.close(grid_fig)
print(grid_path)
show_png(grid_path)


D:\Geometry\artifacts\chapter-02\figures\bivector-drawing-grid\bivector-grid.png


## Bivector Addition And The Shape That Is Not There

When two bivectors have the same attitude, addition is just signed area addition. The drawn parallelograms can be sheared until matching edges line up, but that physical reshaping is only a visual story for the algebraic fact that the element has no preferred boundary. In 3-D, two planes through the origin share at least one line, so many bivector sums can be imagined by factoring out a common direction and reducing the problem to vector addition of the remaining factors. This is why 3-D bivectors are all blades.

The important caveat arrives as soon as the vector space has four dimensions. A general sum of basis bivectors may no longer be factorable as `a wedge b`; it is then a 2-vector but not a 2-blade. This is not pedantry. A blade represents one flat subspace. A nonblade k-vector can be algebraically valid while having no direct interpretation as one k-dimensional subspace in the vector-space model. Later models use four and five algebraic dimensions to compute with three-dimensional geometry, so the blade versus k-vector distinction is not optional background trivia. It is part of the type discipline of geometric algebra.


## Area Ratios Solve 2-D Linear Equations

If `a` and `b` span a plane, the coefficients in `x = alpha a + beta b` can be found by wedging with the opposite basis vector. The repeated term disappears, leaving a ratio of two bivectors in the same plane. In coordinates this agrees with Cramer's rule, but the geometric reading is cleaner: compare oriented areas.


In [5]:
basis_a = np.array([1.2, 0.35])
basis_b = np.array([-0.25, 1.05])
target_x = np.array([0.55, 1.15])

solved = ch2.solve_in_basis_2d(target_x, basis_a, basis_b)
print(json.dumps({k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in solved.items()}, indent=2))
print("reconstruction error:", np.linalg.norm(solved["reconstructed"] - target_x))

assert np.allclose(solved["reconstructed"], target_x)


{
  "alpha": 0.6419294990723563,
  "beta": 0.8812615955473099,
  "reconstructed": [
    0.55,
    1.1500000000000001
  ]
}
reconstruction error: 2.220446049250313e-16


## Planar Line Intersection From Bivector Ratios

The same area-ratio idea finds the intersection of two offset lines in the plane. Choose one point and one direction for each line. The unknown point is reconstructed from the two direction vectors, while the needed bivectors are obtained from the known anchors by reshapeability.


In [6]:
p = np.array([1.0, 0.0])
u = np.array([0.0, 1.0])
q = np.array([0.0, 1.0])
v = np.array([1.0, 1.0])

line_fig, intersection = ch2.line_intersection_figure(p, u, q, v)
line_path = save_matplotlib(
    line_fig,
    "figures",
    "line-intersection",
    "line-intersection.png",
    root=ARTIFACT_ROOT,
    dpi=170,
)
plt.close(line_fig)

print(intersection)
show_png(line_path, width=760)
assert np.allclose(intersection["point"], np.array([1.0, 2.0]))


{'point': array([1., 2.]), 'u_coeff': 1.0, 'v_coeff': 1.0}


## Scalars And The Direct Representation Of Subspaces

The pattern extends downward as well as upward: a scalar can be treated as a grade-0 blade, so scalar multiplication is the outer product with a grade-0 element. This removes the artificial split between "numbers outside the space" and "subspace elements inside the algebra".

For positive grade blades, a direct representation uses containment by wedging: a vector `x` lies in the subspace represented by blade `A` exactly when `x wedge A = 0`. This is a powerful idea, but it has a sharp edge: `A wedge B = 0` only proves the two blades share at least one direction, not that all of `A` is contained in `B`.


In [7]:
scalar = algebra3.scalar(2.5)
assert scalar.wedge(e1).almost_equal(2.5 * e1)
assert scalar.wedge(algebra3.scalar(-2)).almost_equal(algebra3.scalar(-5.0))

xy_plane = e1.wedge(e2)
inside = algebra3.vector([1.7, -0.2, 0.0])
outside = algebra3.vector([1.0, 1.0, 0.5])

print("inside wedge xy_plane =", inside.wedge(xy_plane))
print("outside wedge xy_plane =", outside.wedge(xy_plane))

assert inside.wedge(xy_plane).almost_equal(zero3)
assert not outside.wedge(xy_plane).almost_equal(zero3)

xz_plane = e1.wedge(e3)
print("xz_plane wedge xy_plane =", xz_plane.wedge(xy_plane))
print("e3 wedge xy_plane =", e3.wedge(xy_plane))

assert xz_plane.wedge(xy_plane).almost_equal(zero3)
assert not e3.wedge(xy_plane).almost_equal(zero3)


inside wedge xy_plane = 0
outside wedge xy_plane = 0.5e1^e2^e3
xz_plane wedge xy_plane = 0
e3 wedge xy_plane = e1^e2^e3


## Why Direct Representation Feels Like Incidence

The containment equation `x wedge A = 0` is the chapter's first taste of incidence geometry. A vector lies in the line represented by `a` when the pair fails to span area. A vector lies in the plane represented by `a wedge b` when the triple fails to span volume. The test generalizes cleanly: if wedging a candidate vector onto a blade does not raise the grade, the candidate direction was already present in the represented subspace.

This is also where zero needs to be treated gently. The zero element can be the zero scalar, zero vector, zero bivector, or zero trivector depending on the computation that produced it. Algebraically there is no benefit in splitting it into separate typed zeros inside this compact exterior algebra. Geometrically it represents an empty attempt to span the requested grade: no line from a zero vector, no plane from parallel vectors, no volume from coplanar vectors, and no 4-volume inside an ordinary 3-D vector space.


## The Grassmann Ladder

In an `n`-dimensional vector space, grade `k` has `binomial(n, k)` basis blades. Those counts form Pascal's triangle. A highest-grade blade is a pseudoscalar for the space: every other full-volume element is a scalar multiple of it.


In [8]:
ladder = ch2.basis_blade_counts(5)
ladder_table = pd.DataFrame(
    {f"grade {k}": [row[k] if k < len(row) else "" for row in ladder] for k in range(6)},
    index=[f"n={n}" for n in range(6)],
)
display(ladder_table)

ladder_fig = ch2.pascal_ladder_figure(max_dimension=5)
ladder_path = save_matplotlib(
    ladder_fig,
    "figures",
    "grassmann-ladder",
    "grassmann-ladder.png",
    root=ARTIFACT_ROOT,
    dpi=170,
)
plt.close(ladder_fig)
show_png(ladder_path, width=760)


,grade 0,grade 1,grade 2,grade 3,grade 4,grade 5
n=0,1,,,,,
n=1,1,1,,,,
n=2,1,2,1,,,
n=3,1,3,3,1,,
n=4,1,4,6,4,1,
n=5,1,5,10,10,5,1


## Blades Versus General k-Vectors

Every k-blade is a k-vector, but not every k-vector is a blade. The first unavoidable example appears in four dimensions: `e1 wedge e2 + e3 wedge e4` is a perfectly valid 2-vector, yet it does not represent one plane. One quick diagnostic is containment: a blade that represents a plane should contain a 2-D family of vectors, while this 2-vector contains no nonzero vector.


In [9]:
algebra4 = Algebra([1, 1, 1, 1], names=["e1", "e2", "e3", "e4"])
f1, f2, f3, f4 = algebra4.basis()
B4 = f1.wedge(f2) + f3.wedge(f4)

trivector_masks = sorted({mask for basis_vector in (f1, f2, f3, f4) for mask in basis_vector.wedge(B4).terms})
containment_matrix = np.column_stack(
    [[basis_vector.wedge(B4).terms.get(mask, 0.0) for mask in trivector_masks] for basis_vector in (f1, f2, f3, f4)]
)
rank = np.linalg.matrix_rank(containment_matrix)
nullity = containment_matrix.shape[1] - rank

print("B4 =", B4)
print("basis trivector masks:", [algebra4.basis_name(mask) for mask in trivector_masks])
print(containment_matrix)
print(f"rank={rank}, nullity={nullity}")

assert rank == 4
assert nullity == 0


B4 = e1^e2 + e3^e4
basis trivector masks: ['e1^e2^e3', 'e1^e2^e4', 'e1^e3^e4', 'e2^e3^e4']
[[0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]
rank=4, nullity=0


## Nonmetric Size Is Still Useful

It is tempting to think that the word area must imply a Euclidean measurement. Chapter 2 is more precise than that. The outer product gives relative weights inside the same full-grade carrier. A triangle's oriented area in a chosen plane can be compared with another triangle in that same plane because both are scalar multiples of the same unit area element. A tetrahedron's oriented volume in a chosen 3-D space can be compared with another tetrahedron in that same space for the same reason. What the chapter refuses to do is compare an area in one plane with an area in a different tilted plane before a metric says how those carriers relate.

That distinction is practical. The linear-equation and line-intersection formulas above use ratios of bivectors that already live in a common plane, so the division is meaningful as a scalar ratio even though this notebook has not introduced blade inverses as a general algebraic operation. Chapter 6 will make division native through the geometric product. For now, the ratio notation records the geometric fact that two same-attitude full-grade elements differ by a scalar.


## Reversion And Grade Involution

Two sign-changing operations become useful later. Reversion reverses the order of vector factors in a blade; grade involution flips odd grades. They are simple to compute once the grade is known, and both extend linearly to multivectors.


In [10]:
sign_rows = []
for grade in range(8):
    reversion_sign = (-1) ** (grade * (grade - 1) // 2)
    involution_sign = (-1) ** grade
    sign_rows.append({"grade": grade, "reversion": reversion_sign, "grade_involution": involution_sign})
display(pd.DataFrame(sign_rows))

A = algebra3.scalar(1.0) + 2.0 * e1 + 0.5 * e2.wedge(e3)
C = -0.25 * e2 + 1.5 * e1.wedge(e3) + 0.75 * e1.wedge(e2).wedge(e3)

assert A.wedge(C).reverse().almost_equal(C.reverse().wedge(A.reverse()))
assert A.wedge(C).grade_involution().almost_equal(A.grade_involution().wedge(C.grade_involution()))
print("A wedge C =", A.wedge(C))
print("reverse(A wedge C) =", A.wedge(C).reverse())
print("grade involution(A wedge C) =", A.wedge(C).grade_involution())


,grade,reversion,grade_involution
0,0,1,1
1,1,1,-1
2,2,-1,1
3,3,-1,-1
4,4,1,1
5,5,1,-1
6,6,-1,1
7,7,-1,-1


A wedge C = -0.25e2 - 0.5e1^e2 + 1.5e1^e3 + 0.75e1^e2^e3
reverse(A wedge C) = -0.25e2 + 0.5e1^e2 - 1.5e1^e3 - 0.75e1^e2^e3
grade involution(A wedge C) = 0.25e2 - 0.5e1^e2 + 1.5e1^e3 - 0.75e1^e2^e3


## Programming Lab 1: Drawing Bivectors

The drawing lab above used a fixed vector and a rotating second vector. As the second vector crosses parallel alignment, the signed area passes through zero and changes orientation. This is the computational version of the chapter's point: the sign and weight of an oriented area element are not decoration; they are part of the object.


In [11]:
angles = np.linspace(0, 2 * np.pi, 9)
bivector_samples = []
for theta in angles:
    v_theta = np.array([np.cos(theta), np.sin(theta)])
    bivector_samples.append({"theta": theta, "e12": ch2.wedge2(np.array([1.0, 0.0]), v_theta)})
display(pd.DataFrame(bivector_samples))


,theta,e12
0,0.000000,0.000000e+00
1,0.785398,7.071068e-01
2,1.570796,1.000000e+00
3,2.356194,7.071068e-01
4,3.141593,1.224647e-16
5,3.926991,-7.071068e-01
6,4.712389,-1.000000e+00
7,5.497787,-7.071068e-01
8,6.283185,-2.449294e-16


## Programming Lab 2: Backface Culling

A projected triangle is front-facing or back-facing according to the sign of the oriented area `(b - a) wedge (c - a)`. The artifact below projects a small torus mesh into 2-D and lets you toggle all faces, front faces only, and back faces only. This is the same exterior-algebra test that powers a classic hidden-surface-removal pass.


In [12]:
vertices3d, faces = ch2.torus_mesh(major_count=14, minor_count=7)
vertices2d, depth = ch2.project_vertices(vertices3d)
front = ch2.front_face_mask(vertices2d, faces)

cull_fig = ch2.backface_culling_figure(vertices2d, faces)
cull_path = save_plotly_html(
    cull_fig,
    "labs",
    "backface-culling",
    "backface-culling.html",
    root=ARTIFACT_ROOT,
)
print({"front_faces": int(front.sum()), "back_faces": int((~front).sum()), "total_faces": int(len(front))})
show_html(cull_path, height=780)

assert front.any()
assert (~front).any()


{'front_faces': 98, 'back_faces': 98, 'total_faces': 196}


## Implementation Notes For The Labs

The labs intentionally keep two layers separate. The algebraic layer asks for the sign and grade of an outer product. The drawing layer turns those values into patches, projected triangles, and sampled volumes. In production code those layers often get tangled: a normal vector gets used where an oriented area was meant, a face orientation test silently changes sign after projection, or a vector-field classifier treats a local magnitude test as if it were a topological wrapping test. Chapter 2 gives a small antidote to that habit. Write the spanning test first, then decide how to draw or optimize it.

The backface-culling lab is the most direct programming payoff. The projected vertices of a triangle define two edge vectors, and their 2-D outer product is positive or negative depending on screen orientation. No normal vector is needed in the projected plane. The singularity lab is subtler: each normalized field triangle contributes a trivector, and the sum estimates how much of the unit sphere the field directions wrap. This is a topological signal expressed with oriented volume, not merely a search for small vector magnitudes.


## Programming Lab 3: Singularity Detection With Trivectors

For an isolated zero of a 3-D vector field, normalize field samples on a cube surface and connect neighboring samples into triangles. Each triangle contributes a small trivector volume. If the sampled directions wrap the unit sphere once, the accumulated volume is close to the volume of that sphere; if there is no singularity inside, opposite orientations cancel.

This is a numerical lab, so the score is approximate. Increasing the grid improves the radial point-singularity score, while an empty cube stays near zero.


In [13]:
point_scores = []
for grid in [3, 4, 5, 6, 8]:
    inside_score = ch2.singularity_score(lambda point: ch2.radial_field(point), np.zeros(3), grid=grid)
    outside_score = ch2.singularity_score(lambda point: ch2.radial_field(point), np.array([2.2, 0.0, 0.0]), grid=grid)
    point_scores.append({"grid": grid, "cube_contains_zero": inside_score, "empty_cube": outside_score})
display(pd.DataFrame(point_scores))

assert point_scores[-1]["cube_contains_zero"] > 0.97
assert abs(point_scores[-1]["empty_cube"]) < 0.01


,grid,cube_contains_zero,empty_cube
0,3,0.879388,0.002114
1,4,0.928838,0.001212
2,5,0.953343,0.000783
3,6,0.967165,0.000546
4,8,0.981280,0.000309


The next artifact samples a vector field whose zero set follows a helix. The marker size is based on a local trivector score, while the black curve shows the known zero set. A full curve-singularity classifier needs more topology than this point-index score, but the plot is a useful bridge between the chapter's trivector volume test and the richer examples later in the book.


In [14]:
near_helix = ch2.helix_points(-2.2, 2.2, 11)
offset_helix = near_helix + np.array([1.2, 0.0, 0.0])
centers = np.vstack([near_helix, offset_helix])
helix_scores = ch2.scan_singularity_scores(ch2.helix_gradient_field, centers, half_width=0.35, grid=4)

helix_fig = ch2.singularity_scan_figure(centers, helix_scores)
helix_path = save_plotly_html(
    helix_fig,
    "labs",
    "singularity-scan",
    "singularity-scan.html",
    root=ARTIFACT_ROOT,
)
score_table = pd.DataFrame(
    {
        "x": centers[:, 0],
        "y": centers[:, 1],
        "z": centers[:, 2],
        "score": helix_scores,
        "near_known_zero_set": [True] * len(near_helix) + [False] * len(offset_helix),
    }
)
display(score_table.head())
show_html(helix_path)


,x,y,z,score,near_known_zero_set
0,-0.588501,-0.808496,-2.20,-0.004473,True
1,-0.188077,-0.982154,-1.76,-0.010818,True
2,0.248175,-0.968715,-1.32,-0.006390,True
3,0.637151,-0.770739,-0.88,-0.002308,True
4,0.904752,-0.425939,-0.44,0.000078,True


## Final Invariant Checks

The closing cell gathers the notebook's invariants in one JSON artifact. These checks are intentionally small and structural: antisymmetry, self-wedge zero, bivector-ratio reconstruction, line intersection, point-singularity separation, direct containment, Grassmann nonblade containment, and the monadic operation laws.


## What To Carry Into Chapter 3

The outer product has now built a ladder: scalars, vectors, bivectors, trivectors, and higher blades when the ambient dimension allows them. It has also separated three ideas that everyday vector algebra tends to blur. First, a subspace can be represented directly as an algebraic object. Second, orientation is not an afterthought; it controls signs and therefore computational behavior. Third, grade is a structural invariant, not a display choice.

Chapter 3 will add metric products so that different attitudes can be compared. That is when lengths of different lines, angles between subspaces, perpendicular complements, projections, and dual representations become available. The best preparation is to keep the Chapter 2 invariant in your head: the outer product answers the spanning question. When a later operation measures, projects, or dualizes, ask which part came from spanning and which part came from the metric. That habit prevents many of the classic early geometric-algebra mistakes.


In [15]:
report = ch2.chapter_invariant_report()
report.update(source_span)
report.update(
    {
        "direct_plane_contains_inside_vector": inside.wedge(xy_plane).almost_equal(zero3),
        "direct_plane_rejects_outside_vector": not outside.wedge(xy_plane).almost_equal(zero3),
        "nonblade_B4_contains_nonzero_vector": bool(nullity > 0),
        "reversion_anti_involution_check": A.wedge(C).reverse().almost_equal(C.reverse().wedge(A.reverse())),
        "grade_involution_homomorphism_check": A.wedge(C).grade_involution().almost_equal(
            A.grade_involution().wedge(C.grade_involution())
        ),
        "artifact_root": str(ARTIFACT_ROOT),
    }
)

assert report["wedge2_antisymmetry"]
assert report["wedge2_self_zero"]
assert report["basis_reconstruction_error"] < 1e-12
assert report["line_intersection_error"] < 1e-12
assert report["radial_singularity_center_score"] > 0.9
assert abs(report["radial_empty_cube_score"]) < 0.01
assert report["direct_plane_contains_inside_vector"]
assert report["direct_plane_rejects_outside_vector"]
assert report["nonblade_B4_contains_nonzero_vector"] is False
assert report["reversion_anti_involution_check"]
assert report["grade_involution_homomorphism_check"]

checks_path = save_json(report, "checks", None, "invariant-checks.json", root=ARTIFACT_ROOT)
print(json.dumps(report, indent=2, sort_keys=True))
print("wrote", checks_path)


{
  "artifact_root": "D:\\Geometry\\artifacts\\chapter-02",
  "basis_reconstruction_error": 1.1102230246251565e-16,
  "boundary_check": "PDF page 55 starts Chapter 2; PDF page 97 starts Chapter 3.",
  "direct_plane_contains_inside_vector": true,
  "direct_plane_rejects_outside_vector": true,
  "grade_involution_homomorphism_check": true,
  "line_intersection_error": 0.0,
  "nonblade_B4_contains_nonzero_vector": false,
  "pdf_pages": "55-96",
  "printed_pages": "23-64",
  "radial_empty_cube_score": 0.0007829049350749403,
  "radial_singularity_center_score": 0.9533428804610679,
  "reversion_anti_involution_check": true,
  "wedge2_antisymmetry": true,
  "wedge2_self_zero": true
}
wrote D:\Geometry\artifacts\chapter-02\checks\invariant-checks.json
